In [ ]:
# ============================================
# SETUP & IMPORTS
# ============================================

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import json

from generative.networks.nets import DiffusionModelUNet
from generative.networks.schedulers import DDPMScheduler, DDIMScheduler

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 📊 Quick Start: Class-Conditional Model

We'll start simple - just condition on **pathology type** (which we already have from folder names!)

Later we can add:
- Location (frontal, temporal, parietal, etc.)
- Severity (0-1 scale)
- Patient age
- Lesion size

In [ ]:
# ============================================
# CONFIGURATION - CLASS CONDITIONAL
# ============================================

config = {
    # Data
    'data_dir': 'extracted_data/CT SCANS/Computed Tomography (CT) of the Brain/Computed Tomography (CT) of the Brain/data',
    'image_size': 64,
    'batch_size': 8,
    
    # Model
    'in_channels': 3,
    'out_channels': 3,
    'num_classes': 3,  # aneurysm, cancer, tumor (we'll exclude 'normal' for now)
    'class_conditional': True,  # ← KEY: Enable class conditioning
    
    # Training
    'num_epochs': 100,
    'learning_rate': 1e-4,
    'num_train_timesteps': 1000,
    
    # Generation
    'num_inference_steps': 200,
}

# Class mapping
CLASS_MAP = {
    'aneurysm': 0,
    'cancer': 1,
    'tumor': 2,
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================
# CLASS-CONDITIONAL DATASET
# ============================================

class ClassConditionalBrainCT(Dataset):
    """
    Dataset that returns (image, class_label)
    Class label is extracted from folder name
    """
    def __init__(self, data_dir, transform=None, class_map=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.class_map = class_map or CLASS_MAP
        
        # Collect all images with their class labels
        self.samples = []
        for class_name, class_idx in self.class_map.items():
            class_dir = self.data_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.png'):
                    self.samples.append((img_path, class_idx))
        
        print(f"Found {len(self.samples)} images across {len(self.class_map)} classes")
        
        # Print class distribution
        class_counts = {name: 0 for name in self.class_map.keys()}
        for _, class_idx in self.samples:
            class_name = [k for k, v in self.class_map.items() if v == class_idx][0]
            class_counts[class_name] += 1
        
        print("\nClass distribution:")
        for class_name, count in class_counts.items():
            print(f"  {class_name}: {count} images")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, class_idx = self.samples[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(class_idx, dtype=torch.long)


# Transforms
train_transforms = transforms.Compose([
    transforms.Resize(config['image_size']),
    transforms.CenterCrop(config['image_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

# Create dataset
dataset = ClassConditionalBrainCT(
    data_dir=config['data_dir'],
    transform=train_transforms,
    class_map=CLASS_MAP
)

dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print(f"\nDataLoader: {len(dataloader)} batches per epoch")

In [ ]:
# ============================================
# VISUALIZE SAMPLES WITH CLASS LABELS
# ============================================

# Get a batch
images, labels = next(iter(dataloader))

# Reverse class map for display
idx_to_class = {v: k for k, v in CLASS_MAP.items()}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(min(8, len(images))):
    img = images[i].permute(1, 2, 0).numpy()
    img = (img + 1) / 2  # Denormalize
    
    class_name = idx_to_class[labels[i].item()]
    
    axes[i].imshow(img)
    axes[i].set_title(f"Class: {class_name.upper()}", fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Images with Class Labels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Class labels are correctly extracted from folder names!")

In [ ]:
# ============================================
# CREATE CLASS-CONDITIONAL MODEL
# ============================================

model = DiffusionModelUNet(
    spatial_dims=2,
    in_channels=config['in_channels'],
    out_channels=config['out_channels'],
    num_channels=(128, 256, 512),
    attention_levels=(False, True, True),
    num_res_blocks=2,
    num_head_channels=64,
    num_class_embeds=config['num_classes'],  # ← KEY: Enable class conditioning
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print(f"MODEL: CLASS-CONDITIONAL DIFFUSION UNET")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / (1024**2):.1f} MB")
print(f"\n✅ Model is CLASS-CONDITIONAL (can control pathology type!)")
print(f"{'='*60}")

# Schedulers
scheduler = DDPMScheduler(num_train_timesteps=config['num_train_timesteps'])
gen_scheduler = DDIMScheduler(num_train_timesteps=config['num_train_timesteps'])

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])

print("\n✅ Setup complete! Ready to train.")

In [ ]:
# ============================================
# TRAINING: CLASS-CONDITIONAL DIFFUSION
# ============================================

print(f"\n{'='*60}")
print(f"🏥 TRAINING CLASS-CONDITIONAL BRAIN CT DIFFUSION MODEL")
print(f"{'='*60}")
print(f"Epochs: {config['num_epochs']}")
print(f"Batch size: {config['batch_size']}")
print(f"Learning rate: {config['learning_rate']}")
print(f"Classes: {list(CLASS_MAP.keys())}")
print(f"{'='*60}\n")

scaler = GradScaler('cuda')
losses = []
best_loss = float('inf')

for epoch in range(config['num_epochs']):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
    for images, class_labels in pbar:
        images = images.to(device)
        class_labels = class_labels.to(device)
        
        optimizer.zero_grad()
        
        # Random timesteps
        timesteps = torch.randint(
            0, config['num_train_timesteps'], (images.shape[0],)
        ).to(device)
        
        # Add noise
        noise = torch.randn_like(images)
        noisy_images = scheduler.add_noise(images, noise, timesteps)
        
        # Predict noise WITH class conditioning
        with autocast('cuda'):
            noise_pred = model(
                noisy_images,
                timesteps=timesteps,
                class_labels=class_labels  # ← CLASS CONDITIONING HERE!
            )
            loss = nn.functional.mse_loss(noise_pred, noise)
        
        # Backward
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    # Epoch stats
    epoch_loss = total_loss / len(dataloader)
    losses.append(epoch_loss)
    lr_scheduler.step()
    
    # Save best model
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
            'config': config,
            'class_map': CLASS_MAP,
        }, 'brain_ct_class_conditional_model.pth')
        print(f"✅ Saved best model at epoch {epoch+1}")
    
    print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss:.6f} | Best: {best_loss:.6f} | LR: {lr_scheduler.get_last_lr()[0]:.2e}")
    
    # Generate samples every 25 epochs
    if (epoch + 1) % 25 == 0:
        model.eval()
        gen_scheduler.set_timesteps(50)
        
        # Generate one sample per class
        with torch.no_grad():
            noise = torch.randn(config['num_classes'], 3, config['image_size'], config['image_size']).to(device)
            class_labels_gen = torch.arange(config['num_classes']).to(device)
            
            image = noise
            for t in gen_scheduler.timesteps:
                noise_pred = model(
                    image,
                    timesteps=torch.tensor([t] * config['num_classes']).to(device),
                    class_labels=class_labels_gen
                )
                image, _ = gen_scheduler.step(noise_pred, t, image)
            
            # Display
            fig, axes = plt.subplots(1, config['num_classes'], figsize=(4*config['num_classes'], 4))
            idx_to_class = {v: k for k, v in CLASS_MAP.items()}
            
            for i in range(config['num_classes']):
                img = image[i].cpu().permute(1, 2, 0).numpy()
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
                
                axes[i].imshow(img)
                axes[i].set_title(f"{idx_to_class[i].upper()}", fontsize=12, fontweight='bold')
                axes[i].axis('off')
            
            plt.suptitle(f'Generated Samples - Epoch {epoch+1}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'generated_images/epoch_{epoch+1}_class_conditional.png', dpi=150, bbox_inches='tight')
            plt.show()

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"Best loss: {best_loss:.6f}")
print(f"Model saved: brain_ct_class_conditional_model.pth")
print(f"{'='*60}")

In [ ]:
# ============================================
# GENERATE: Specific Pathology Type
# ============================================

def generate_pathology(pathology_type, num_samples=4):
    """
    Generate CT scans for a specific pathology type
    
    Args:
        pathology_type: 'aneurysm', 'cancer', or 'tumor'
        num_samples: Number of samples to generate
    """
    print(f"\n{'='*60}")
    print(f"🏥 GENERATING: {pathology_type.upper()}")
    print(f"{'='*60}")
    
    model.eval()
    gen_scheduler.set_timesteps(200)
    
    # Get class label
    class_idx = CLASS_MAP[pathology_type]
    class_labels = torch.tensor([class_idx] * num_samples).to(device)
    
    # Generate
    noise = torch.randn(num_samples, 3, config['image_size'], config['image_size']).to(device)
    image = noise
    
    with torch.no_grad():
        for t in tqdm(gen_scheduler.timesteps, desc="Generating"):
            noise_pred = model(
                image,
                timesteps=torch.tensor([t] * num_samples).to(device),
                class_labels=class_labels
            )
            image, _ = gen_scheduler.step(noise_pred, t, image)
    
    # Display
    fig, axes = plt.subplots(1, num_samples, figsize=(4*num_samples, 4))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        img = image[i].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        
        axes[i].imshow(img)
        axes[i].set_title(f'Sample {i+1}', fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle(f'Generated {pathology_type.upper()} Scans', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return image.cpu()


# Generate examples
tumor_scans = generate_pathology('tumor', num_samples=4)
aneurysm_scans = generate_pathology('aneurysm', num_samples=4)
cancer_scans = generate_pathology('cancer', num_samples=4)

# 🎯 Next Steps: Full Patient Conditioning

Once this class-conditional model works, we can extend it to include:

1. **Location** (frontal, temporal, parietal, occipital)
2. **Severity** (0-1 scale)
3. **Patient age** (normalized)
4. **Lesion size** (in mm)

This requires:
- Creating metadata JSON/CSV file
- Modifying dataset to return condition vectors
- Using the `PatientConditionedUNet` from the BrainCT_Model notebook

---

## Download Better Dataset (Optional):

**BraTS 2020** has full tumor location masks + metadata:
- 3,000+ brain MRI scans
- Tumor segmentation masks (exact voxel-level location!)
- Patient age, survival data, tumor grade
- Download: https://www.med.upenn.edu/cbica/brats2020/registration.html

With BraTS, you could train a model that:
```python
generate_scan(
    pathology='glioblastoma',
    location='right_frontal',
    tumor_size=35,  # mm
    age=45,
    severity=0.8
)
```

And it would generate a scan showing a glioblastoma in the right frontal lobe!